# Chapter 2: Kinematics & Coordinate Transformations

In Chapter 1, we established the 6-DOF state vectors ($\eta$ for map position, $\nu$ for vehicle speed). Now, we look at **Kinematics**—the branch of mechanics that describes motion without considering the forces that cause it. 

The fundamental problem of marine kinematics is bridging the gap between two different worlds: **The Map** and **The Robot**.

---

## 1. The Two Worlds: Reference Frames

To map out a vehicle's path across an ocean, we must constantly juggle two different coordinate systems:

1. **The Earth-Fixed Frame (NED):**
   * This is your **Map Space**. It is anchored to the Earth.
   * The axes are fixed: **$+X$ points North**, **$+Y$ points East**, and **$+Z$ points Down** (toward the center of the Earth).
   * Position ($\eta$) is always calculated here.

2. **The Body-Fixed Frame (BODY):**
   * This is your **Vehicle Space**. It is glued to the vessel and moves *with* it.
   * The axes turn as the boat turns: **$+X$ is Forward (Surge)**, **$+Y$ is Starboard (Sway)**, and **$+Z$ is Down (Heave)**.
   * Velocities ($\nu$) are measured here because onboard sensors (like IMUs or speed logs) only know what the boat is physically feeling.

---

## 2. The Mathematical Bridge: The Rotation Matrix

If a robotic boat faces dead North, moving "Forward" increases its North coordinate on the map. But what if it spins 90 degrees to face East? Now, moving "Forward" increases its East coordinate on the map.

To automate this translation, we use a **Rotation Matrix** ($R_b^n$). It acts as a mathematical lens that takes velocities from the **BODY** frame ($u, v$) and rotates them into the **NED** map frame ($\dot{N}, \dot{E}$):

$$\begin{bmatrix} \dot{N} \\ \dot{E} \end{bmatrix} = R_b^n(\psi) \begin{bmatrix} u \\ v \end{bmatrix}$$

For a flat, 2D surface, this rotation matrix uses basic right-triangle trigonometry:

$$R_b^n(\psi) = \begin{bmatrix} \cos(\psi) & -\sin(\psi) \\ \sin(\psi) & \cos(\psi) \end{bmatrix}$$

Where $\psi$ (Psi) is the vessel's heading angle relative to true North. Let's use the interactive workshop below to see exactly how this trigonometry manipulates vectors in real time!

In [1]:
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from ipywidgets import interact

def visualize_kinematics(heading_deg, thrust_forward, thrust_sideways):
    """
    Computes the 2D rotation matrix mapping BODY velocities to NED coordinates
    and displays the resulting vector breakdown.
    """
    # Convert heading from compass degrees to radians
    psi = np.radians(heading_deg)
    
    # 1. Compute the 2D Rotation Matrix R_b_n
    R_b_n = np.array([
        [np.cos(psi), -np.sin(psi)],
        [np.sin(psi),  np.cos(psi)]
    ])
    
    # 2. Bundle the BODY frame velocities [u, v]
    V_body = np.array([thrust_forward, thrust_sideways])
    
    # 3. Transform to NED map frame using matrix dot product
    V_ned = R_b_n.dot(V_body)
    
    # --- Plotting Assembly ---
    fig, ax = plt.subplots(figsize=(8, 8))
    
    # Draw fixed Earth map grid (NED axes)
    ax.axhline(0, color='black', linewidth=1, alpha=0.4)
    ax.axvline(0, color='black', linewidth=1, alpha=0.4)
    
    # Plot the resulting world velocity vector (NED motion)
    ax.quiver(0, 0, V_ned[1], V_ned[0], angles='xy', scale_units='xy', scale=1, 
              color='crimson', label='Actual Motion on Map (NED Vector)', zorder=3)
    
    # Visual indicator of the vessel hull orientation
    hull_length = 1.2
    vessel_dx = hull_length * np.cos(psi)
    vessel_dy = hull_length * np.sin(psi)
    # Note: Map X is North (up), Map Y is East (right)
    ax.plot([-vessel_dy, vessel_dy], [-vessel_dx, vessel_dx], color='royalblue', 
            linewidth=6, label='Vessel Hull Orientation', alpha=0.7)
    ax.plot(vessel_dy, vessel_dx, 'ro', markersize=8, label='Bow (Front)') 
    
    # Live data output display panel
    text_info = (
        f"Heading (ψ): {heading_deg}° ({psi:.3f} rad)\n\n"
        f"Rotation Matrix R_b_n:\n"
        f"  [  {R_b_n[0,0]:.3f}   {R_b_n[0,1]:.3f} ]\n"
        f"  [  {R_b_n[1,0]:.3f}    {R_b_n[1,1]:.3f} ]\n\n"
        f"Map Velocity Vector Output:\n"
        f"  North Speed (N_dot) = {V_ned[0]:.2f} m/s\n"
        f"  East Speed (E_dot)  = {V_ned[1]:.2f} m/s"
    )
    ax.text(-4.5, 2.2, text_info, fontsize=11, family='monospace',
            bbox=dict(boxstyle='round', facecolor='whitesmoke', edgecolor='gray', alpha=0.9))
    
    # Plot constraints
    ax.set_xlim([-5, 5])
    ax.set_ylim([-5, 5])
    ax.set_xlabel('EAST (Meters/Second Map Displacement)', fontsize=10)
    ax.set_ylabel('NORTH (Meters/Second Map Displacement)', fontsize=10)
    ax.set_title('Kinematic Workshop: Transforming BODY to NED Space', fontsize=12)
    ax.grid(True, linestyle=':', alpha=0.5)
    ax.legend(loc='lower right')
    
    plt.show()

print("MANIPULATE THE VEHICLE FRAME TO TRANSFORM THE VECTORS:")
interact(visualize_kinematics,
         heading_deg=widgets.FloatSlider(min=0.0, max=360.0, step=5.0, value=0.0, description='Heading (ψ):'),
         thrust_forward=widgets.FloatSlider(min=-3.0, max=3.0, step=0.2, value=2.0, description='Surge (u):'),
         thrust_sideways=widgets.FloatSlider(min=-3.0, max=3.0, step=0.2, value=0.0, description='Sway (v):'));

MANIPULATE THE VEHICLE FRAME TO TRANSFORM THE VECTORS:


interactive(children=(FloatSlider(value=0.0, description='Heading (ψ):', max=360.0, step=5.0), FloatSlider(val…

### Kinematic Lab Challenges to Try:

1. **The Identity Frame Alignment:**
   Set the `Heading` slider to $0^\circ$ and `Surge (u)` to 2.0. Look at the values in the rotation matrix box. It reads `[ 1, 0 ] / [ 0, 1 ]`. This is an **Identity Matrix**. Because the vessel is facing dead North, its internal forward speed translates directly into pure Northward map velocity.

2. **The Pure $90^\circ$ Orthogonal Shift:**
   Rotate the `Heading` slider to $90^\circ$ (facing East), and keep `Surge (u)` at 2.0. Notice that even though the vehicle is only executing a "forward" surge command in its own mind, the crimson arrow on the map points completely along the **East** axis. Look at the matrix: the coefficients shifted to route the surge velocity straight to the East channel.

3. **Why this matters for SeaPath:**
   When you implement your real-world state estimator, your hardware will read angular rates from an IMU and velocity values from an odometer or Doppler Velocity Log (DVL). If your code miscalculates this matrix conversion by even a single degree or slips a negative sign, your automated trajectory tracking loop will instantly steering the craft into a completely different quadrant of the map.

---

## 3. Scaling to 3D: The Full Principal Rotation Matrices

In a true 3-dimensional marine environment (especially for submarines, underwater drones, or surface ships crashing through heavy seas), a simple 2D matrix isn't enough. We have to handle three separate rotations, known as **Euler Angles**:
* **Roll ($\phi$):** Tilting side-to-side around the longitudinal axis.
* **Pitch ($\theta$):** Tilting nose-up/nose-down around the transverse axis.
* **Yaw ($\psi$):** Spinning compass heading around the vertical axis.

Fossen breaks full 3D rotation down into three individual **Principal Rotation Matrices**, one for each axis:

$$R_{x,\phi} = \begin{bmatrix} 1 & 0 & 0 \\ 0 & \cos(\phi) & -\sin(\phi) \\ 0 & \sin(\phi) & \cos(\phi) \end{bmatrix}, \quad R_{y,\theta} = \begin{bmatrix} \cos(\theta) & 0 & \sin(\theta) \\ 0 & 1 & 0 \\ -\sin(\theta) & 0 & \cos(\theta) \end{bmatrix}, \quad R_{z,\psi} = \begin{bmatrix} \cos(\psi) & -\sin(\psi) & 0 \\ \sin(\psi) & \cos(\psi) & 0 \\ 0 & 0 & 1 \end{bmatrix}$$

To get the final, complete rotation matrix ($R_b^n$), we multiply them together in a strict sequence (**zyx order**):

$$R_b^n = R_{z,\psi} \, R_{y,\theta} \, R_{x,\phi}$$

---

## 4. The Mathematical Trap: Gimbal Lock

While Euler angles are incredibly intuitive for humans to read, they contain a fatal mathematical flaw when used in 3D computer models. 

When you multiply these three matrices together, look closely at what happens if the vessel pitches straight up at a **$90^\circ$ angle ($\theta = \pm90^\circ$)**. The trigonometry collapses, and the equations for **Roll ($\phi$)** and **Yaw ($\psi$)** become identical. 

The computer literally loses a degree of freedom. It can no longer tell the difference between rolling side-to-side or spinning its compass heading. This mathematical catastrophe is called **Gimbal Lock**.

Let's run the diagnostic tool below to watch the math break down when a craft points straight up!

In [2]:
def analyze_gimbal_lock(roll_deg, pitch_deg, yaw_deg):
    """
    Computes the full 3D Euler rotation matrix and checks for Gimbal Lock
    by monitoring the alignment of the rotational transformation channels.
    """
    # Convert inputs to radians
    phi = np.radians(roll_deg)
    theta = np.radians(pitch_deg)
    psi = np.radians(yaw_deg)
    
    # Principal Rotations
    Rx = np.array([[1, 0, 0],
                   [0, np.cos(phi), -np.sin(phi)],
                   [0, np.sin(phi),  np.cos(phi)]])
    
    Ry = np.array([[np.cos(theta), 0, np.sin(theta)],
                   [0, 1, 0],
                   [-np.sin(theta), 0, np.cos(theta)]])
    
    Rz = np.array([[np.cos(psi), -np.sin(psi), 0],
                   [np.sin(psi),  np.cos(psi), 0],
                   [0, 0, 1]])
    
    # Combined Rotation Matrix (R = Rz * Ry * Rx)
    R_b_n = Rz.dot(Ry).dot(Rx)
    
    print("-" * 60)
    print("                 3D ROTATION MATRIX R_b_n")
    print("-" * 60)
    print(f"  [  {R_b_n[0,0]:.3f}   {R_b_n[0,1]:.3f}   {R_b_n[0,2]:.3f} ]")
    print(f"  [  {R_b_n[1,0]:.3f}   {R_b_n[1,1]:.3f}   {R_b_n[1,2]:.3f} ]")
    print(f"  [  {R_b_n[2,0]:.3f}   {R_b_n[2,1]:.3f}   {R_b_n[2,2]:.3f} ]")
    print("-" * 60)
    
    # Check for Gimbal Lock status
    # If cos(pitch) is close to 0, pitch is near +/- 90 degrees
    is_locked = np.isclose(np.abs(np.cos(theta)), 0.0, atol=1e-3)
    
    if is_locked:
        print("⚠️  STATUS: GIMBAL LOCK DETECTED! ⚠️")
        print("Notice that changing Roll or Yaw now manipulates the exact same matrix elements.")
        print("The system has lost an independent rotational axis.")
    else:
        print("✅ STATUS: SYSTEM SAFE (Axes are mathematically independent)")

# Create interactive sliders for 3D exploration
interact(analyze_gimbal_lock,
         roll_deg=widgets.FloatSlider(min=-180.0, max=180.0, step=5.0, value=0.0, description='Roll (φ):'),
         pitch_deg=widgets.FloatSlider(min=-90.0, max=90.0, step=1.0, value=0.0, description='Pitch (θ):'),
         yaw_deg=widgets.FloatSlider(min=-180.0, max=180.0, step=5.0, value=0.0, description='Yaw (ψ):'));

interactive(children=(FloatSlider(value=0.0, description='Roll (φ):', max=180.0, min=-180.0, step=5.0), FloatS…

### Kinematics Takeaway: Why We Need Quaternions

1. **The 90-Degree Breakdown:**
   Leave `Roll` and `Yaw` at 0, and slowly slide the `Pitch (θ)` slider up to exactly `90°`. Watch the status message change to **GIMBAL LOCK DETECTED**. 
   
2. **Losing a Axis:**
   With the pitch locked at 90°, wiggle the `Roll` slider, then wiggle the `Yaw` slider. Notice that both sliders modify the exact same positions inside the matrix. The numbers move identically because the physical axes have lined up on top of one another, paralyzing the computer's orientation tracking.

3. **The GNC Engineering Solution:**
   For an autopilot tracking a standard surface ship, a pitch of 90° means the boat is literally pointing vertical like a rocket—an impossible scenario. Because of this, marine engineers safely use Euler angles for standard ships. 
   
   However, for advanced underwater robotics (AUVs) or highly maneuverable submarines that can pull steep vertical dives, Gimbal Lock will completely crash the navigation software. To solve this, advanced frameworks bypass Euler angles entirely and use a 4-dimensional hyper-complex number system called **Quaternions** ($q = [q_0, q_1, q_2, q_3]^T$) to track spatial rotation with zero singularities.

In [3]:
def quaternion_demonstration(pitch_deg):
    """
    Demonstrates how Euler angles translate to a Quaternion representation,
    proving how the scalar and vector parts remain stable even at a 90-degree pitch.
    """
    # For this demo, let's lock Roll and Yaw to 0 to watch Pitch do its work
    phi = 0.0
    theta = np.radians(pitch_deg)
    psi = 0.0
    
    # 1. Compute Euler Rotation Matrix (the vulnerable way)
    # R = Rz(0) * Ry(theta) * Rx(0) -> simplifies directly to the Y-axis rotation
    R_euler = np.array([
        [np.cos(theta),  0, np.sin(theta)],
        [0,              1, 0],
        [-np.sin(theta), 0, np.cos(theta)]
    ])
    
    # 2. Compute the Quaternion elements (The Stable Way)
    # For a pure Pitch (rotation around the Y-axis), the formulas simplify to:
    q0 = np.cos(theta / 2.0)
    q1 = 0.0
    q2 = np.sin(theta / 2.0)
    q3 = 0.0
    
    # Normalize to ensure unit length
    mag = np.sqrt(q0**2 + q1**2 + q2**2 + q3**2)
    q0, q1, q2, q3 = q0/mag, q1/mag, q2/mag, q3/mag
    
    print("=" * 60)
    print(f"               EULER ANGLE APPARATUS (Pitch = {pitch_deg}°)")
    print("=" * 60)
    print(f"  Rotation Matrix Column 1 (X-Axis Mapping): [{R_euler[0,0]:.3f}, {R_euler[1,0]:.3f}, {R_euler[2,0]:.3f}]")
    print(f"  Rotation Matrix Column 3 (Z-Axis Mapping): [{R_euler[0,2]:.3f}, {R_euler[1,2]:.3f}, {R_euler[2,2]:.3f}]")
    
    print("\n" + "=" * 60)
    print("               4-DIMENSIONAL QUATERNION VECTOR (q)")
    print("=" * 60)
    print(f"  q = [  q0: {q0:.3f}  |  q1: {q1:.3f}  ,  q2: {q2:.3f}  ,  q3: {q3:.3f}  ]")
    print("-" * 60)
    print(f"  Scalar Part (q0) = {q0:.3f}  (Tracks the magnitude of rotation)")
    print(f"  Vector Part (i,j,k) = [{q1:.1f}, {q2:.3f}, {q3:.1f}] (Defines the 3D Rotation Axis)")
    
    if np.isclose(pitch_deg, 90.0) or np.isclose(pitch_deg, -90.0):
        print("\n🔥 OBSERVATION FOR THE CRITICAL 90° POINT:")
        print(" Look at the Quaternion! The numbers are perfectly distinct, clean fractions.")
        print(" There are no overlapping trigonometric dependencies or ambiguous channels.")
        print(" Your C++ state estimator can ingest this vector safely with zero math loops freezing.")
    else:
        print("\n✅ System status: Linear tracking vector active.")

# Slider to transition into the pitch singularity zone
interact(quaternion_demonstration,
         pitch_deg=widgets.FloatSlider(min=0.0, max=90.0, step=5.0, value=0.0, description='Pitch (θ):'));

interactive(children=(FloatSlider(value=0.0, description='Pitch (θ):', max=90.0, step=5.0), Output()), _dom_cl…

---

## 5. The Gyro Transformation ($T_b^n$)

We know how to rotate linear velocities using $R_b^n$. But what about angular speeds? If an onboard gyroscope measures a turning rate ($p, q, r$) inside the hull, can we just use the same rotation matrix to find how fast our compass heading ($\dot{\psi}$) is changing?

**No.** Angular rates do not transform the same way linear velocities do because Euler angles are nested inside one another. Instead, Fossen defines an **Angular Transformation Matrix** ($T_\Theta(\eta)$):

$$\dot{\Theta} = \begin{bmatrix} \dot{\phi} \\ \dot{\theta} \\ \dot{\psi} \end{bmatrix} = T_\Theta(\eta) \begin{bmatrix} p \\ q \\ r \end{bmatrix}$$

Where the transformation matrix is structured as:

$$T_\Theta(\eta) = \begin{bmatrix} 1 & \sin(\phi)\tan(\theta) & \cos(\phi)\tan(\theta) \\ 0 & \cos(\phi) & -\sin(\phi) \\ 0 & \sin(\phi)/\cos(\theta) & \cos(\phi)/\cos(\theta) \end{bmatrix}$$

Look closely at the denominators in that matrix: they are divided by **$\cos(\theta)$**. If your Pitch ($\theta$) hits $90^\circ$, $\cos(90^\circ) = 0$. You are suddenly dividing by zero! This proves that the angular velocity math *also* completely breaks down at the exact same Euler singularity point.

---

## 6. Lever Arm Compensation (Rigid-Body Velocity Shift)

In an ideal world, your GPS antenna and Doppler Velocity Log (DVL) would be glued exactly to the vessel's Center of Gravity (CG). In reality, the GPS is up on a mast, and the DVL is down on the hull.

If the boat is spinning, a sensor mounted away from the center of gravity will experience an extra, deceptive velocity vector due to the rotation. To calculate the true velocity at the center of gravity ($v_g$) from a distant sensor ($v_s$), we use the cross product of the angular rate vector ($\omega$) and the **lever arm offset vector** ($r_s$):

$$v_g = v_s - (\omega \times r_s)$$

Let's run a quick Python check below to see how a spinning ship creates ghost velocities on a mast-mounted sensor!

In [4]:
def compute_lever_arm(yaw_rate_deg_sec, antenna_offset_x):
    """
    Calculates the ghost velocity felt by a GPS antenna mounted away from
    the vessel's center of gravity during a sharp turn.
    """
    # Angular velocity vector [p, q, r] - assuming pure flat yaw rate
    r_rad = np.radians(yaw_rate_deg_sec)
    omega = np.array([0.0, 0.0, r_rad])
    
    # Lever arm vector [x, y, z] from CG to the antenna (meters forward)
    r_sensor = np.array([antenna_offset_x, 0.0, -5.0]) # 5 meters up on a mast
    
    # Compute the velocity distortion: omega x r_sensor
    v_distortion = np.cross(omega, r_sensor)
    
    print("=" * 60)
    print("             LEVER ARM VELOCITY DISTORTION")
    print("=" * 60)
    print(f"  Vessel Turning Rate : {yaw_rate_deg_sec}°/sec ({r_rad:.3f} rad/s)")
    print(f"  Antenna Location    : {antenna_offset_x}m Forward of Center of Gravity")
    print("-" * 60)
    print(f"  Ghost Sway Velocity Induced at Sensor (v) = {v_distortion[1]:.3f} m/s")
    print("-" * 60)
    print("💡 GNC ARCHITECTURE NOTE:")
    print("  If the vessel is spinning rapidly, your GPS will report that the ship is")
    print("  sliding sideways even if the hull's true center of gravity is tracing a")
    print("  perfectly straight track. Your SeaPath estimator must subtract this exact")
    print("  distortion vector in real-time to prevent navigation loop divergence.")

interact(compute_lever_arm,
         yaw_rate_deg_sec=widgets.FloatSlider(min=0.0, max=30.0, step=1.0, value=10.0, description='Yaw Rate (r):'),
         antenna_offset_x=widgets.FloatSlider(min=0.0, max=50.0, step=1.0, value=15.0, description='Antenna Link (m):'));

interactive(children=(FloatSlider(value=10.0, description='Yaw Rate (r):', max=30.0, step=1.0), FloatSlider(va…

### Chapter 2 Lab Summary & Kinematic Takeaways

1. **The 2D Vector Translation:**
   When the vessel is aligned with the map frame ($\psi = 0^\circ$), the rotation matrix is an Identity matrix, and body velocities translate 1:1 to map movements. As the heading rotates, the matrix automatically routes the vehicle's internal velocities into the correct North and East coordinates using standard trigonometry.

2. **The Euler Pitch Singularity ($90^\circ$ Collapse):**
   By setting your Pitch slider to exactly $90^\circ$, you witnessed **Gimbal Lock** first-hand. Because the middle matrix (Pitch) rotated perpendicular to the horizon, the outer ring (Yaw) and inner ring (Roll) physically aligned. The matrix columns became identical, meaning the computer lost an entire degree of freedom and could no longer tell a roll apart from a compass turn.

3. **The Quaternion Rescue:**
   When you looked at the Quaternion vector at that exact same $90^\circ$ pitch point, the numbers remained beautifully stable, clean fractions (`q = [0.707, 0.0, 0.707, 0.0]`). Because a Quaternion defines orientation using a single 3D vector axis and a single angle of rotation, it has no sequential "middle rings" to collapse.

4. **The Gyro Transformation Trap ($T_b^n$):**
   We discovered that angular rates ($p, q, r$) do not transform the same way linear velocities do. The matrix used to convert gyro rates to map angle rates ($T_b^n$) explicitly divides by $\cos(\theta)$. When pitching to $90^\circ$, this triggers a critical **division-by-zero error**, proving that Euler angle tracking breaks down mathematically for both position *and* rotation.

5. **Lever Arm Realities ($\omega \times r_s$):**
   Sensors are rarely located exactly at the vessel's Center of Gravity (CG). When the vessel rotates, any off-center sensor (like a GPS antenna up on a mast) experiences a "ghost" velocity caused by the rotation lever arm. To maintain sub-meter accuracy, navigation software must calculate and subtract this cross-product vector ($\omega \times r_s$) in real-time.